# Notebook 1: Five models — three skill-alias strategies
This notebook evaluates five sentence-transformer models with three representations of each ESCO skill: **first**, **mean**, and **max** aliases.

All 15 configurations use the same development queries, complete candidate-skill corpus, qrels, metrics, and scoring procedure. One JSON result is written for each configuration, plus a consolidated CSV for the next notebook.

The next notebook will select one global alias strategy using mean graded nDCG across the five models. This notebook records all results without selecting a winner.

In [6]:
import ast
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

ROOT = Path.cwd()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
if not (ROOT / "ranking_metrics.py").exists():
    raise FileNotFoundError(f"Repository root not found from {Path.cwd()}")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from ranking_metrics import evaluate_query

VAL_DIR = ROOT / "data" / "raw" / "TaskB" / "validation"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

MODELS = {
    "minilm": {"name": "all-MiniLM-L6-v2", "query_prefix": "", "skill_prefix": ""},
    "mpnet": {"name": "all-mpnet-base-v2", "query_prefix": "", "skill_prefix": ""},
    "jobbert": {"name": "TechWolf/JobBERT-v2", "query_prefix": "", "skill_prefix": ""},
    "bge": {
        "name": "BAAI/bge-base-en-v1.5",
        "query_prefix": "Represent this sentence for searching relevant skills: ",
        "skill_prefix": "",
    },
    "e5": {
        "name": "intfloat/e5-base-v2",
        "query_prefix": "query: ",
        "skill_prefix": "passage: ",
    },
}
ALIAS_STRATEGIES = ("first", "mean", "max")
print(f"Python: {sys.executable}")
print(f"Repository: {ROOT}")
print(f"Configurations: {len(MODELS) * len(ALIAS_STRATEGIES)}")

Python: d:\apps\conda\envs\job-skill-matching\python.exe
Repository: d:\tuke\spj\Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches
Configurations: 15


## Load the fixed development evaluation set

The candidate corpus is not reduced to sampled training negatives. Every job ranks the complete validation skill corpus, and the raw qrels provide graded relevance (`0`, `1` contextual, `2` core).

In [7]:
queries = pd.read_csv(VAL_DIR / "queries", sep="\t")
corpus = pd.read_csv(VAL_DIR / "corpus_elements", sep="\t")
qrels = pd.read_csv(
    VAL_DIR / "qrels.tsv",
    sep="\t",
    header=None,
    names=["q_id", "iteration", "c_id", "relevance"],
)
corpus["skill_aliases"] = corpus["skill_aliases"].map(ast.literal_eval)
corpus["skill_aliases"] = corpus["skill_aliases"].map(
    lambda aliases: [str(alias).strip() for alias in aliases if str(alias).strip()]
)
if corpus["skill_aliases"].map(len).eq(0).any():
    raise ValueError("The validation corpus contains a skill without aliases")

skill_ids = corpus["c_id"].tolist()
skill_index = {skill_id: index for index, skill_id in enumerate(skill_ids)}
grades_by_job = {}
for job_id, job_qrels in qrels.groupby("q_id", sort=False):
    grades = np.zeros(len(skill_ids), dtype=np.float32)
    for row in job_qrels.itertuples(index=False):
        if row.c_id in skill_index:
            grades[skill_index[row.c_id]] = row.relevance
    grades_by_job[job_id] = grades
query_text_by_id = dict(zip(queries["q_id"], queries["jobtitle"]))
job_ids = [job_id for job_id in queries["q_id"] if job_id in grades_by_job]
if not job_ids:
    raise ValueError("No evaluation queries have qrels")

print(f"queries: {len(job_ids)}")
print(f"candidate skills: {len(corpus)}")
print(f"qrels: {len(qrels)}")
print(f"relevant qrels: {(qrels['relevance'] > 0).sum()}")

queries: 304
candidate skills: 9052
qrels: 56418
relevant qrels: 56418


## Shared encoding and evaluation functions

Embeddings are normalized before cosine scoring. For `max`, aliases are kept separate and the maximum similarity is used per jobâ€“skill pair. This is different from averaging aliases before scoring.

In [8]:
def encode_texts(model, texts, prefix="", batch_size=64):
    prepared = [prefix + str(text) for text in texts]
    return model.encode(
        prepared,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)


def encode_skill_representations(model, specification):
    aliases = corpus["skill_aliases"].tolist()
    prefix = specification["skill_prefix"]
    if specification["alias_strategy"] == "first":
        return {"mode": "matrix", "embeddings": encode_texts(model, [items[0] for items in aliases], prefix)}
    if specification["alias_strategy"] == "mean":
        counts = [len(items) for items in aliases]
        flattened = encode_texts(model, [alias for items in aliases for alias in items], prefix)
        averaged = []
        start = 0
        for count in counts:
            vector = flattened[start : start + count].mean(axis=0)
            vector /= max(np.linalg.norm(vector), 1e-12)
            averaged.append(vector)
            start += count
        return {"mode": "matrix", "embeddings": np.asarray(averaged, dtype=np.float32)}
    if specification["alias_strategy"] == "max":
        flattened = encode_texts(model, [alias for items in aliases for alias in items], prefix)
        return {"mode": "aliases", "embeddings": flattened, "counts": [len(items) for items in aliases]}
    raise ValueError(f"Unknown alias strategy: {specification['alias_strategy']}")


def evaluate_scores(job_embeddings, skill_representations):
    per_query = []
    if skill_representations["mode"] == "matrix":
        score_matrix = job_embeddings @ skill_representations["embeddings"].T
        for row_index, job_id in enumerate(job_ids):
            per_query.append(evaluate_query(score_matrix[row_index], grades_by_job[job_id]))
        return per_query

    alias_embeddings = skill_representations["embeddings"]
    counts = skill_representations["counts"]
    starts = np.cumsum([0] + counts[:-1])
    for row_index, job_id in enumerate(job_ids):
        similarities = job_embeddings[row_index] @ alias_embeddings.T
        skill_scores = np.asarray(
            [similarities[start : start + count].max() for start, count in zip(starts, counts)],
            dtype=np.float32,
        )
        per_query.append(evaluate_query(skill_scores, grades_by_job[job_id]))
    return per_query


def save_configuration_result(short_name, specification, per_query):
    metric_names = per_query[0].keys()
    metrics = {name: float(np.mean([row[name] for row in per_query])) for name in metric_names}
    result = {
        "description": (
            f"Zero-shot cosine ({specification['name']}); "
            f"{specification['alias_strategy']} skill aliases"
        ),
        "model_short_name": short_name,
        "model_name": specification["name"],
        "skill_representation": specification["alias_strategy"],
        "query_prefix": specification["query_prefix"],
        "skill_prefix": specification["skill_prefix"],
        "n_queries": len(job_ids),
        "n_candidate_skills": len(skill_ids),
        **{name: round(value, 6) for name, value in metrics.items()},
    }
    path = RESULTS_DIR / f"cosine_{short_name}_{specification['alias_strategy']}.json"
    path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved: {path}")
    return result

## Run all 15 configurations

Each model is loaded once. Its job embeddings are reused across the three alias strategies; skill embeddings are recomputed per strategy because `mean` and `max` require different representations.

In [9]:
all_results = []
for short_name, model_config in MODELS.items():
    print(f"\n===== {short_name}: {model_config['name']} =====")
    model = SentenceTransformer(model_config["name"])
    job_embeddings = encode_texts(
        model,
        [query_text_by_id[job_id] for job_id in job_ids],
        model_config["query_prefix"],
    )
    for alias_strategy in ALIAS_STRATEGIES:
        print(f"\n--- alias strategy: {alias_strategy} ---")
        specification = {**model_config, "alias_strategy": alias_strategy}
        skill_representations = encode_skill_representations(model, specification)
        per_query = evaluate_scores(job_embeddings, skill_representations)
        all_results.append(save_configuration_result(short_name, specification, per_query))
    del model

comparison = (
    pd.DataFrame(all_results)
    .sort_values(["ndcg_graded", "map_binary"], ascending=False)
    .reset_index(drop=True)
)
comparison_path = RESULTS_DIR / "model_alias_comparison_15.csv"
comparison.to_csv(comparison_path, index=False)
display(comparison[[
    "model_short_name", "model_name", "skill_representation",
    "ndcg_graded", "ndcg_binary", "map_binary", "mrr_binary",
    "p@1", "p@5", "p@10",
]])
print(f"\nSaved consolidated comparison: {comparison_path}")
print(f"JSON configurations written: {len(all_results)}")


===== minilm: all-MiniLM-L6-v2 =====


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]


--- alias strategy: first ---


Batches:   0%|          | 0/142 [00:00<?, ?it/s]

...

Saved: d:\tuke\spj\Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches\results\cosine_e5_mean.json

--- alias strategy: max ---


Batches:   0%|          | 0/1006 [00:00<?, ?it/s]

Saved: d:\tuke\spj\Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches\results\cosine_e5_max.json


,model_short_name,model_name,skill_representation,ndcg_graded,ndcg_binary,map_binary,mrr_binary,p@1,p@5,p@10
0,jobbert,TechWolf/JobBERT-v2,mean,0.738077,0.788888,0.341614,0.788649,0.648026,0.562500,0.517763
1,jobbert,TechWolf/JobBERT-v2,first,0.735927,0.789857,0.347853,0.821024,0.697368,0.564474,0.491447
2,jobbert,TechWolf/JobBERT-v2,max,0.722241,0.772278,0.307647,0.751939,0.595395,0.513158,0.481579
3,e5,intfloat/e5-base-v2,first,0.649977,0.694880,0.183272,0.709678,0.549342,0.467105,0.438816
4,e5,intfloat/e5-base-v2,max,0.644426,0.688065,0.174700,0.671313,0.506579,0.432895,0.399671
5,e5,intfloat/e5-base-v2,mean,0.640110,0.682140,0.164249,0.695816,0.542763,0.450658,0.416776
6,mpnet,all-mpnet-base-v2,mean,0.632662,0.676252,0.161164,0.630215,0.453947,0.398026,0.371053
7,bge,BAAI/bge-base-en-v1.5,mean,0.630855,0.672003,0.155516,0.634001,0.444079,0.413816,0.373684
8,mpnet,all-mpnet-base-v2,first,0.625080,0.670368,0.154499,0.613456,0.440789,0.380921,0.354934
9,bge,BAAI/bge-base-en-v1.5,first,0.624571,0.668177,0.151286,0.650626,0.486842,0.392763,0.365132



Saved consolidated comparison: d:\tuke\spj\Job-Skill-Matching-Ranking-and-Classification-Using-NLP-Approaches\results\model_alias_comparison_15.csv
JSON configurations written: 15


## Alias-strategy summary for Notebook 2

The following table is descriptive only. Notebook 2 should lock the strategy with the highest mean graded nDCG across the five models, using MAP as a tie-breaker.

In [ ]:
alias_summary = (
    comparison.groupby("skill_representation", as_index=False)
    .agg(
        mean_ndcg_graded=("ndcg_graded", "mean"),
        mean_map_binary=("map_binary", "mean"),
        mean_mrr_binary=("mrr_binary", "mean"),
    )
    .sort_values(["mean_ndcg_graded", "mean_map_binary"], ascending=False)
    .reset_index(drop=True)
)
display(alias_summary)
alias_summary.to_csv(RESULTS_DIR / "alias_strategy_summary.csv", index=False)
print("Saved alias summary: results/alias_strategy_summary.csv")

,skill_representation,mean_ndcg_graded,mean_map_binary,mean_mrr_binary
0,mean,0.652597,0.194036,0.676697
1,first,0.650188,0.195984,0.683031
2,max,0.642592,0.180798,0.647566


Saved alias summary: results/alias_strategy_summary.csv


: 